# Fine-tune IndicTrans2 for English → Sanskrit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brandenleelundy-cpu/sanskrit-translate/blob/main/training/finetune_en_sa.ipynb)

This notebook follows the plan's gates:

1. **Baseline (Gate 1)**: score the current model on two held-out test sets.
2. **Fine-tune** on cleaned English–Sanskrit pairs.
3. **Compare (Gate 2)**: score again; only push if it's a real win.
4. **Publish** to the Hugging Face Hub, then point the Space at it with `MODEL_EN_SA`.

**Before you run it**

- Runtime: **GPU** (Colab: *Runtime → Change runtime type → T4 GPU*; Kaggle: *Accelerator → GPU T4 x2*, and turn *Internet* on).
- While signed in to Hugging Face, accept the terms on these pages (one click each, approved instantly):
  [BPCC](https://huggingface.co/datasets/ai4bharat/BPCC) ·
  [IN22-Gen](https://huggingface.co/datasets/ai4bharat/IN22-Gen) ·
  the base model ([200M](https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M) / [1B](https://huggingface.co/ai4bharat/indictrans2-en-indic-1B)).
- Make a **Write** token at huggingface.co/settings/tokens and add it as a secret named `HF_TOKEN`
  (Colab: the 🔑 icon in the left bar, and allow notebook access; Kaggle: *Add-ons → Secrets*).

Then **Run all**. With the default settings it takes roughly 1–1.5 hours on a free T4.

In [ ]:
# 1. Install. Versions are pinned: IndicTrans2's custom model code breaks on newer transformers.
!pip -q install "transformers==4.46.3" "accelerate==1.1.1" "peft==0.13.2" "datasets==3.1.0" \
    sentencepiece sacremoses sacrebleu IndicTransToolkit pandas pyarrow

In [ ]:
# 2. Settings: the only cell you should need to edit.

BASE_MODEL = "ai4bharat/indictrans2-en-indic-dist-200M"  # fast on the free CPU Space
# BASE_MODEL = "ai4bharat/indictrans2-en-indic-1B"       # better, slower; set USE_LORA = True with it
USE_LORA = False

HUB_REPO = "bll1984/indictrans2-en-sa-ft"   # where the fine-tuned model is published
PRIVATE = False                              # a private repo needs a token in the Space that can read it

# Training data (English-Sanskrit). BPCC files are human translations from AI4Bharat.
BPCC_FILES = ["bpcc-seed-latest/san_Deva.tsv", "daily/san_Deva.tsv", "comparable/san_Deva.tsv"]
INCLUDE_BPCC_MINED = False   # + nllb_filtered (web-mined, noisier)
BPCC_MINED_MAX = 50_000
# Itihasa: Ramayana/Mahabharata verses. Its Sanskrit side is classical verse, so too much of it
# teaches the model to answer modern English in shloka style. Keep it a minority.
ITIHASA_MAX = 10_000

EPOCHS = 2
BATCH_SIZE = 16
GRAD_ACCUM = 2
LEARNING_RATE = 5e-4 if USE_LORA else 3e-5
MAX_LEN = 256
NUM_BEAMS = 5
EVAL_ITIHASA_N = 500      # Itihasa test sentences to score (general-domain IN22-Gen is always scored in full)
MIN_GAIN_CHRF = 0.5       # Gate 2: required chrF++ gain on IN22-Gen before pushing
FORCE_PUSH = False        # push even if Gate 2 fails
SEED = 42

SRC, TGT = "eng_Latn", "san_Deva"

In [ ]:
# 3. Sign in to Hugging Face (reads the HF_TOKEN secret; falls back to a prompt).
import os
from huggingface_hub import login

token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if token:
    login(token=token)
else:
    login()  # paste a Write token when asked

import torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to a GPU first."
device = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 4. Helpers: column detection and cleaning.
import random, re, unicodedata
import pandas as pd

random.seed(SEED)

def _ratio(s, pred):
    chars = [c for c in str(s) if not c.isspace()]
    return sum(map(pred, chars)) / len(chars) if chars else 0.0

def deva_ratio(s):  return _ratio(s, lambda c: "ऀ" <= c <= "ॿ")
def latin_ratio(s): return _ratio(s, lambda c: c.isascii())

def pick_columns(df):
    # Find the English and Sanskrit columns, whatever they're called.
    cols = [str(c) for c in df.columns]
    for e, s in [("eng_Latn", "san_Deva"), ("src", "tgt"), ("en", "sa"), ("english", "sanskrit")]:
        if e in cols and s in cols:
            return e, s
    eng = [c for c in cols if "eng" in c.lower()]
    san = [c for c in cols if "san" in c.lower()]
    if len(eng) == 1 and len(san) == 1:
        return eng[0], san[0]
    assert len(cols) <= 4, f"Ambiguous columns {cols}: set them by hand"
    sample = df.head(300).astype(str)
    deva = {c: sample[c].map(deva_ratio).mean() for c in sample.columns}
    lat  = {c: sample[c].map(latin_ratio).mean() for c in sample.columns}
    s_col = max(deva, key=deva.get)
    e_col = max((c for c in lat if c != s_col), key=lat.get)
    assert deva[s_col] > 0.5 and lat[e_col] > 0.5, f"Couldn't find columns in {cols}"
    return e_col, s_col

def norm(s):
    return " ".join(unicodedata.normalize("NFC", str(s)).split())

def clean_pairs(pairs, exclude_en=frozenset()):
    seen, out = set(), []
    for en, sa in pairs:
        en, sa = norm(en), norm(sa)
        if not en or not sa or en.lower() in exclude_en:
            continue
        ne, ns = len(en.split()), len(sa.split())
        if ne > 120 or ns > 120:
            continue
        r = len(sa) / max(1, len(en))
        if r < 0.3 or r > 3.0:
            continue
        if deva_ratio(sa) < 0.6 or latin_ratio(en) < 0.7:
            continue
        key = (en.lower(), sa)
        if key in seen:
            continue
        seen.add(key)
        out.append((en, sa))
    return out

In [ ]:
# 5. Load test sets first, so no test sentence leaks into training.
from huggingface_hub import hf_hub_download, list_repo_files

# IN22-Gen: 1024 general-domain sentences (news, culture, legal, ...), expert-translated.
in22_path = hf_hub_download("ai4bharat/IN22-Gen", "data/train-00000-of-00001.parquet", repo_type="dataset")
in22 = pd.read_parquet(in22_path)
e, s = pick_columns(in22)
test_in22 = [(norm(a), norm(b)) for a, b in zip(in22[e], in22[s])]
print("IN22-Gen test:", len(test_in22), "| columns used:", e, s)

# Itihasa (verse) — test split.
ITI_REV = "refs/convert/parquet"
def itihasa(split):
    p = hf_hub_download("rahular/itihasa", f"Itihasa/{split}/0000.parquet", repo_type="dataset", revision=ITI_REV)
    df = pd.read_parquet(p)
    return [(norm(t["en"]), norm(t["sn"])) for t in df["translation"]]

iti_test = itihasa("test")
random.shuffle(iti_test)
test_iti = iti_test[:EVAL_ITIHASA_N]
print("Itihasa test:", len(test_iti))

exclude = frozenset(en.lower() for en, _ in test_in22 + iti_test)

In [ ]:
# 6. Load and clean training data.
def read_bpcc(path):
    p = hf_hub_download("ai4bharat/BPCC", path, repo_type="dataset")
    df = pd.read_csv(p, sep="\t", header=None, quoting=3, on_bad_lines="skip", dtype=str).dropna()
    e, s = pick_columns(df)
    pairs = list(zip(df[e], df[s]))
    # drop a header row if the file had one
    if pairs and re.fullmatch(r"[A-Za-z_]+", str(pairs[0][0])) and re.fullmatch(r"[A-Za-z_]+", str(pairs[0][1])):
        pairs = pairs[1:]
    return pairs

raw = {}
for f in BPCC_FILES:
    raw[f] = read_bpcc(f)
if INCLUDE_BPCC_MINED:
    mined = read_bpcc("nllb_filtered/san_Deva.tsv")
    random.shuffle(mined)
    raw["nllb_filtered (mined)"] = mined[:BPCC_MINED_MAX]
iti_train = itihasa("train") + itihasa("validation")
random.shuffle(iti_train)
raw["itihasa"] = iti_train[:ITIHASA_MAX]

train_pairs = []
for name, pairs in raw.items():
    kept = clean_pairs(pairs, exclude)
    print(f"{name:32s} raw {len(pairs):>7,}  kept {len(kept):>7,}")
    train_pairs += kept

train_pairs = clean_pairs(train_pairs)   # dedupe across sources
random.shuffle(train_pairs)
dev_pairs, train_pairs = train_pairs[:1000], train_pairs[1000:]
print(f"\nTraining pairs: {len(train_pairs):,}   dev: {len(dev_pairs):,}")
for en, sa in train_pairs[:3]:
    print(" -", en, "=>", sa)

In [ ]:
# 7. Translation + scoring helpers.
import sacrebleu
from IndicTransToolkit import IndicProcessor
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

ip_infer = IndicProcessor(inference=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

def translate(model, sents, bs=16):
    model.eval()
    out = []
    for i in range(0, len(sents), bs):
        batch = ip_infer.preprocess_batch(sents[i:i + bs], src_lang=SRC, tgt_lang=TGT)
        enc = tokenizer(batch, truncation=True, padding="longest", max_length=MAX_LEN,
                        return_tensors="pt", return_attention_mask=True).to(device)
        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
            try:
                gen = model.generate(**enc, num_beams=NUM_BEAMS, max_length=MAX_LEN, use_cache=True)
            except Exception:
                gen = model.generate(**enc, num_beams=NUM_BEAMS, max_length=MAX_LEN, use_cache=False)
        dec = tokenizer.batch_decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        out += ip_infer.postprocess_batch(dec, lang=TGT)
    return out

def score(model, pairs, label):
    hyps = translate(model, [e for e, _ in pairs])
    refs = [s for _, s in pairs]
    chrf = sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score
    bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
    print(f"{label:28s} chrF++ {chrf:6.2f}   BLEU {bleu:6.2f}")
    return {"chrf++": round(chrf, 2), "bleu": round(bleu, 2)}

PROBES = ["Hello", "Where is the train station?", "How much does this cost?",
          "Thank you very much for your help.", "The weather is pleasant today.",
          "Knowledge is the greatest wealth."]

In [ ]:
# 8. GATE 1 — baseline.
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, trust_remote_code=True).to(device)
results = {"base": {"IN22-Gen": score(base_model, test_in22, "base  | IN22-Gen"),
                    "Itihasa": score(base_model, test_iti, "base  | Itihasa")}}
base_probes = translate(base_model, PROBES)
del base_model; torch.cuda.empty_cache()

In [ ]:
# 9. Tokenize training data (IndicTrans2's own recipe: tags on the source, target-vocab labels).
from datasets import Dataset
from tqdm.auto import tqdm

ip_train = IndicProcessor(inference=False)

def prep(pairs, chunk=2000):
    src, tgt = [], []
    for i in tqdm(range(0, len(pairs), chunk), desc="preprocess"):
        c = pairs[i:i + chunk]
        src += ip_train.preprocess_batch([e for e, _ in c], src_lang=SRC, tgt_lang=TGT, is_target=False)
        tgt += ip_train.preprocess_batch([s for _, s in c], src_lang=TGT, tgt_lang=SRC, is_target=True)
    return Dataset.from_dict({"sentence_SRC": src, "sentence_TGT": tgt})

def tok_fn(ex):
    x = tokenizer(ex["sentence_SRC"], truncation=True, padding=False, max_length=MAX_LEN)
    with tokenizer.as_target_tokenizer():
        y = tokenizer(ex["sentence_TGT"], truncation=True, padding=False, max_length=MAX_LEN)
    x["labels"] = y["input_ids"]
    return x

train_ds = prep(train_pairs).map(tok_fn, batched=True, remove_columns=["sentence_SRC", "sentence_TGT"])
dev_ds   = prep(dev_pairs).map(tok_fn, batched=True, remove_columns=["sentence_SRC", "sentence_TGT"])
print(train_ds)

In [ ]:
# 10. Fine-tune.
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, trust_remote_code=True)
if USE_LORA:
    from peft import LoraConfig, get_peft_model
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.1, bias="none", task_type="SEQ_2_SEQ_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]))
    model.print_trainable_parameters()

try:
    from IndicTransToolkit import IndicDataCollator as Collator
except ImportError:
    Collator = DataCollatorForSeq2Seq
collator = Collator(tokenizer=tokenizer, model=model, padding="longest",
                    pad_to_multiple_of=8, label_pad_token_id=-100)

args = Seq2SeqTrainingArguments(
    output_dir="ckpt",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="inverse_sqrt" if USE_LORA else "linear",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,
    group_by_length=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=50,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
)

trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds,
                         eval_dataset=dev_ds, data_collator=collator)
trainer.train()

model = trainer.model
if USE_LORA:
    model = model.merge_and_unload()
model = model.to(device)

In [ ]:
# 11. GATE 2 — compare with the baseline.
results["fine-tuned"] = {"IN22-Gen": score(model, test_in22, "tuned | IN22-Gen"),
                         "Itihasa": score(model, test_iti, "tuned | Itihasa")}
tuned_probes = translate(model, PROBES)

gain = results["fine-tuned"]["IN22-Gen"]["chrf++"] - results["base"]["IN22-Gen"]["chrf++"]
GATE2_PASS = gain >= MIN_GAIN_CHRF
print(f"\nIN22-Gen chrF++ gain: {gain:+.2f}  ->  Gate 2 {'PASSED' if GATE2_PASS else 'NOT passed'}")

print("\nSpot checks (base  →  fine-tuned):")
for p, a, b in zip(PROBES, base_probes, tuned_probes):
    print(f"- {p}\n    {a}\n    {b}")

In [ ]:
# 12. Publish to the Hugging Face Hub (only when Gate 2 passes, unless FORCE_PUSH).
import json, shutil
from huggingface_hub import HfApi, snapshot_download

if not (GATE2_PASS or FORCE_PUSH):
    print("Not pushing: no clear win on general text. Try more data, another epoch, or the 1B base.")
else:
    out = "final_model"
    shutil.rmtree(out, ignore_errors=True)
    # Copy the base repo's tokenizer files and custom model code, then write the new weights over it.
    snapshot_download(BASE_MODEL, local_dir=out,
                      ignore_patterns=["*.safetensors", "*.bin", "*.pt", "*.h5", "*.msgpack", "README.md"])
    model.save_pretrained(out, safe_serialization=True)

    # Smoke test: reload from disk exactly as the Space will.
    chk_tok = AutoTokenizer.from_pretrained(out, trust_remote_code=True)
    chk = AutoModelForSeq2SeqLM.from_pretrained(out, trust_remote_code=True).to(device)
    print("Reload check:", translate(chk, ["Where is the train station?"]))
    del chk

    sources = ", ".join(raw.keys())
    card = f'''---
license: mit
language: [en, sa]
base_model: {BASE_MODEL}
tags: [translation, indictrans2, sanskrit]
---
# IndicTrans2 fine-tuned for English → Sanskrit

Fine-tuned from `{BASE_MODEL}` on {len(train_pairs):,} cleaned pairs ({sources}).

| Test set | Base chrF++ | Fine-tuned chrF++ | Base BLEU | Fine-tuned BLEU |
|---|---|---|---|---|
''' + "\n".join(
        f"| {k} | {results['base'][k]['chrf++']} | {results['fine-tuned'][k]['chrf++']} | "
        f"{results['base'][k]['bleu']} | {results['fine-tuned'][k]['bleu']} |" for k in ["IN22-Gen", "Itihasa"]
    ) + '''

Load it like the base model (`trust_remote_code=True`) and preprocess with IndicTransToolkit's
`IndicProcessor`, using `src_lang="eng_Latn"`, `tgt_lang="san_Deva"`.
'''
    open(f"{out}/README.md", "w").write(card)
    json.dump(results, open(f"{out}/eval_results.json", "w"), indent=2)

    api = HfApi()
    api.create_repo(HUB_REPO, private=PRIVATE, exist_ok=True)
    api.upload_folder(folder_path=out, repo_id=HUB_REPO, commit_message="Fine-tuned English->Sanskrit")
    print(f"\nPublished: https://huggingface.co/{HUB_REPO}")
    print(f"Next: in the Space's Settings set the variable MODEL_EN_SA = {HUB_REPO} and restart it.")

## After it's published

1. In the Space (**aqueous-api → Settings → Variables and secrets**), set `MODEL_EN_SA` to `bll1984/indictrans2-en-sa-ft` and click **Restart Space**.
2. The site picks it up automatically: no change to the website needed.
3. Rolling back is the same switch: delete the variable (or set it to the base model) and restart.

**If Gate 2 doesn't pass**, the usual next moves, in order: try the 1B base with `USE_LORA = True`;
add the mined data (`INCLUDE_BPCC_MINED = True`); add a third epoch; add your own domain sentences
(e.g. phrases your users actually type) as an extra list in step 6.